# Лекция 07. Свёрточные сети / Lecture 07. Convolutional Networks

**Практика: семь тем, после каждой мини-задание 5–10 минут. / Practice: seven topics, each followed by a 5–10 minute mini-task.**

**Данные / Data:** UCI Optical Recognition of Handwritten Digits, E. Alpaydin & C. Kaynak (1998), [DOI:10.24432/C50P49](https://doi.org/10.24432/C50P49), [UCI](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), [sklearn loader](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html). UCI: CC BY 4.0, сохраняйте авторство / retain attribution.

RU: `load_digits` содержит 1797 реальных изображений 8×8 из исходной тестовой части UCI. Наблюдение — рукописная цифра; признаки — 64 агрегированных значения интенсивности 0…16; цель — класс 0…9. Это не фотографии высокого разрешения и не ImageNet. Делим на 16 по известному диапазону. Учебное стратифицированное разбиение seed=707: 1078 train, 359 validation, 360 test. Идентификаторов авторов нет, независимость почерков не гарантируется. Это не официальный benchmark. В предыдущей лекции использовались те же исходные данные; новая разбивка не превращает их в неизвестный внешний тест. Сравнения служат учебной демонстрацией. Test не используем для настройки.

EN: `load_digits` contains 1797 real 8×8 images copied from the original UCI test portion. One observation is a handwritten digit; features are 64 aggregated intensity values in 0…16; the target is a class in 0…9. These are not high-resolution photographs or ImageNet. Divide by the known range 16. Teaching stratified split, seed=707: 1078 train, 359 validation, 360 test. Writer IDs are unavailable; writer independence is not guaranteed. This is not the official benchmark. The previous lecture used the same underlying data; a new split does not make these an unseen external test. Comparisons are teaching demonstrations. Never tune using test.

**Среда / Environment:** Google Colab, CPU; NumPy, scikit-learn, matplotlib, PyTorch. Dataset bundled with sklearn; no synthetic fallback. / Данные входят в sklearn, синтетической замены нет. Run cells top to bottom after Restart / Запускайте сверху вниз после Restart.

## 1. Реальные данные и NCHW / Real data and NCHW

RU: CNN сохраняет двумерную структуру изображения. Канал — отдельная ось даже у полутоновой цифры.

EN: A CNN preserves image geometry. Even a grayscale digit needs a channel axis.

In [ ]:
import copy  # RU: Копирование весов. EN: Copy model weights.
import json  # RU: Сохранение результатов. EN: Save results.
import numpy as np  # RU: Массивы и ручные операции. EN: Arrays and manual operations.
import matplotlib.pyplot as plt  # RU: Строим графики. EN: Draw plots.
import torch  # RU: Тензоры и обучение. EN: Tensors and training.
from sklearn.datasets import load_digits  # RU: Реальные цифры UCI. EN: Real UCI digits.
from sklearn.model_selection import train_test_split  # RU: Стратификация разбиения. EN: Stratified splitting.
from sklearn.metrics import ConfusionMatrixDisplay  # RU: Матрица ошибок. EN: Confusion matrix.
torch.set_num_threads(1)  # RU: Быстрый CPU для малых данных. EN: One CPU thread suits this small dataset.
torch.manual_seed(707)  # RU: Повторяемая инициализация. EN: Reproducible initialization.
digits = load_digits()  # RU: Загружаем 1797 наблюдений. EN: Load 1797 observations.
images = digits.images.astype(np.float32) / 16.0  # RU: Известный диапазон, без подгонки. EN: Known range, no fitted scaling.
labels = digits.target  # RU: Метки 0–9. EN: Labels 0–9.
indices = np.arange(len(labels))  # RU: Индексы исходных наблюдений. EN: Original observation indices.
train_ids, rest_ids = train_test_split(indices, test_size=0.4, stratify=labels, random_state=707)  # RU: Отделяем train. EN: Separate training data.
val_ids, test_ids = train_test_split(rest_ids, test_size=0.5, stratify=labels[rest_ids], random_state=707)  # RU: Разделяем validation и test. EN: Split validation and test.
x = torch.tensor(images[:, None], dtype=torch.float32)  # RU: Добавляем канал: N×1×8×8. EN: Add channel: N×1×8×8.
y = torch.tensor(labels, dtype=torch.long)  # RU: Целые метки для cross-entropy. EN: Integer labels for cross-entropy.
assert not (set(train_ids) & set(val_ids) or set(train_ids) & set(test_ids) or set(val_ids) & set(test_ids))  # RU: Проверяем непересечение. EN: Verify disjoint splits.

print('NCHW', tuple(x.shape), 'split', len(train_ids), len(val_ids), len(test_ids))  # RU: Проверяем формы. EN: Check shapes.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))  # RU: Сетка реальных цифр. EN: Grid of real digits.
for digit, ax in enumerate(axes.flat):  # RU: По одному примеру класса. EN: One example per class.
    index = train_ids[np.flatnonzero(labels[train_ids] == digit)[0]]  # RU: Берём пример из train. EN: Pick a training example.
    ax.imshow(images[index], cmap='gray', vmin=0, vmax=1)  # RU: Фиксируем шкалу яркости. EN: Fix the intensity scale.
    ax.set_title(str(digit))  # RU: Подписываем метку. EN: Label the class.
    ax.axis('off')  # RU: Убираем координаты. EN: Hide coordinates.
plt.show()  # RU: Показываем изображения. EN: Display images.

**Мини-задание / Mini-task · 5–10 min.**
RU: (1) Выведите число объектов каждого класса отдельно для train, validation, test. (2) Проверьте, что каждый исходный индекс встречается ровно в одном разбиении. (3) Для первых семи train-объектов покажите формы до и после добавления канала. (4) Объясните различие NCHW и NHWC. Сдайте таблицу 3×10, три формы и 2–3 предложения. Критерий: сумма таблицы 1797, пересечений нет, вход CNN (7,1,8,8).
EN: (1) Count each class separately in train, validation and test. (2) Verify every original index belongs to exactly one split. (3) Show the shapes of the first seven training images before and after adding a channel. (4) Explain NCHW versus NHWC. Deliver a 3×10 table, three shapes and 2–3 sentences. Acceptance: counts total 1797, no overlap, CNN input (7,1,8,8).

In [ ]:
# Ваше решение / Your solution

## 2. Свёртка вручную / Convolution by hand

RU: В PyTorch Conv2d вычисляет кросс-корреляцию: ядро не переворачивается. Историческое название «свёртка» сохраняем, соглашение фиксируем.

EN: PyTorch Conv2d computes cross-correlation: the kernel is not flipped. We keep the historical term convolution and state the convention.

In [ ]:
def correlate2d(image, kernel, stride=1, padding=0):  # RU: Одноканальная корреляция, ядро не переворачиваем. EN: Single-channel correlation without flipping the kernel.
    padded = np.pad(image, padding, mode='constant')  # RU: Нулевое дополнение. EN: Zero padding.
    kh, kw = kernel.shape  # RU: Размеры ядра. EN: Kernel dimensions.
    oh = (image.shape[0] + 2 * padding - kh) // stride + 1  # RU: Высота результата. EN: Output height.
    ow = (image.shape[1] + 2 * padding - kw) // stride + 1  # RU: Ширина результата. EN: Output width.
    result = np.empty((oh, ow), dtype=np.float64)  # RU: Место для выходной карты. EN: Allocate the output map.
    for row in range(oh):  # RU: Перебираем строки выхода. EN: Iterate over output rows.
        for col in range(ow):  # RU: Перебираем столбцы выхода. EN: Iterate over output columns.
            patch = padded[row * stride:row * stride + kh, col * stride:col * stride + kw]  # RU: Берём локальное окно. EN: Extract the local window.
            result[row, col] = np.sum(patch * kernel)  # RU: Скалярное произведение с общим ядром. EN: Dot product with the shared kernel.
    return result  # RU: Возвращаем карту отклика. EN: Return the response map.
image = images[train_ids[0]]  # RU: Берём реальную train-цифру. EN: Use a real training digit.
kernel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)  # RU: Задаём фиксированный фильтр Собеля. EN: Define a fixed Sobel filter.
response = correlate2d(image, kernel, padding=1)  # RU: Получаем подписанный отклик. EN: Compute the signed response.
reference = torch.nn.functional.conv2d(torch.tensor(image[None, None], dtype=torch.float64), torch.tensor(kernel[None, None]), padding=1)[0, 0].numpy()  # RU: Независимая проверка PyTorch. EN: Independent PyTorch check.
np.testing.assert_allclose(response, reference, atol=1e-12)  # RU: Сравниваем все пиксели. EN: Compare every pixel.
print('max discrepancy', np.abs(response - reference).max())  # RU: Печатаем расхождение. EN: Print the discrepancy.

**Мини-задание / Mini-task · 5–10 min.**
RU: Для выходной позиции (3,3) при padding=1 выпишите окно 3×3, поэлементное произведение и сумму. Сравните эту сумму с `response[3,3]`. Затем примените транспонированное ядро и покажите обе карты с общей симметричной шкалой. Объясните, какие направления изменения яркости выделяются. Сдать: окно, произведение, число, две карты. Проверка: сумма равна выходному пикселю с atol=1e-12.
EN: For output position (3,3) with padding=1, print the 3×3 patch, elementwise product and sum. Compare the sum with `response[3,3]`. Then apply the transposed kernel and display both maps on a common symmetric scale. Explain the brightness-change directions. Deliver the patch, product, scalar and two maps. Acceptance: the sum matches the output pixel within atol=1e-12.

In [ ]:
# Ваше решение / Your solution

## 3. Padding, stride и размер / Padding, stride and shape

RU: Для dilation=1: out = floor((in+2p−k)/s)+1. Формула применяется отдельно к высоте и ширине.

EN: With dilation=1, out = floor((in+2p−k)/s)+1, independently for height and width.

In [ ]:
for stride, padding in [(1, 0), (1, 1), (2, 1)]:  # RU: Сравниваем три настройки. EN: Compare three settings.
    result = correlate2d(image, kernel, stride=stride, padding=padding)  # RU: Фильтруем одно и то же изображение. EN: Filter the same image.
    expected = (8 + 2 * padding - 3) // stride + 1  # RU: Считаем размер по формуле. EN: Compute the formula output size.
    assert result.shape == (expected, expected)  # RU: Проверяем обе оси. EN: Check both axes.
    print(stride, padding, result.shape)  # RU: Выводим размеры. EN: Print the shapes.

**Мини-задание / Mini-task · 5–10 min.**
RU: Возьмите прямоугольный crop `image[:7,:6]`. Для ядра 3×3, stride=2, padding=1 сначала вручную рассчитайте H и W, затем сравните NumPy и `F.conv2d`. Повторите с padding=0. Укажите, какие окна не помещаются и почему не используем округление вверх. Сдать: две предсказанные формы, код проверки и пояснение 2–3 предложения.
EN: Take the rectangular crop `image[:7,:6]`. For a 3×3 kernel, stride=2 and padding=1, calculate H and W by hand before comparing NumPy with `F.conv2d`. Repeat with padding=0. Identify incomplete windows and explain why the formula floors rather than rounds up. Deliver two predicted shapes, verification code and 2–3 sentences.

In [ ]:
# Ваше решение / Your solution

## 4. Pooling и потеря информации / Pooling and information loss

RU: Max pooling выбирает максимум окна. Он не учится и не делает сеть полностью инвариантной.

EN: Max pooling selects a window maximum. It has no trainable weights and does not make the whole network invariant.

In [ ]:
def pool2d(image, size=2, stride=2):  # RU: Max pooling одной карты без padding. EN: Max pooling of one map without padding.
    oh = (image.shape[0] - size) // stride + 1  # RU: Полные окна по высоте. EN: Complete windows along height.
    ow = (image.shape[1] - size) // stride + 1  # RU: Полные окна по ширине. EN: Complete windows along width.
    result = np.empty((oh, ow), dtype=image.dtype)  # RU: Сохраняем тип данных. EN: Preserve the data type.
    for row in range(oh):  # RU: Строка окна. EN: Window row.
        for col in range(ow):  # RU: Столбец окна. EN: Window column.
            result[row, col] = image[row * stride:row * stride + size, col * stride:col * stride + size].max()  # RU: Максимум локального окна. EN: Maximum within the local window.
    return result  # RU: Возвращаем уменьшенную карту. EN: Return the reduced map.
activated = np.maximum(response, 0)  # RU: Применяем ReLU к карте. EN: Apply ReLU to the map.
pooled = pool2d(activated)  # RU: Уменьшаем 8×8 до 4×4. EN: Reduce 8×8 to 4×4.
reference = torch.nn.functional.max_pool2d(torch.tensor(activated[None, None]), 2)[0, 0].numpy()  # RU: Проверка библиотекой. EN: Library check.
np.testing.assert_array_equal(pooled, reference)  # RU: Максимумы должны совпасть. EN: Maxima must match.
print(activated.shape, pooled.shape)  # RU: Показываем изменение формы. EN: Show the shape change.

**Мини-задание / Mini-task · 5–10 min.**
RU: Для той же карты сравните max pooling 2×2 с average pooling 2×2. Нарисуйте исходную карту и оба результата, выпишите первое окно и два агрегата. Найдите окно, где результаты отличаются. Почему восстановить исходную карту по максимумам невозможно? Сдать: три карты, окно, два числа и аргумент.
EN: Compare 2×2 max pooling with 2×2 average pooling on the same map. Plot the original and both outputs, print the first window and its aggregates, and find a window where the outputs differ. Why can the original map not be reconstructed from maxima? Deliver three maps, a window, two numbers and an argument.

In [ ]:
# Ваше решение / Your solution

## 5. CNN: формы и параметры / CNN: shapes and parameters

RU: Два блока Conv–ReLU–Pool и dense-голова. Вход 1×8×8 → 8×8×8 → 8×4×4 → 16×4×4 → 16×2×2 → 64 → 10.

EN: Two Conv–ReLU–Pool blocks and a dense head. Input 1×8×8 → 8×8×8 → 8×4×4 → 16×4×4 → 16×2×2 → 64 → 10.

In [ ]:
def build_cnn():  # RU: Определяем фиксированную архитектуру. EN: Define the fixed architecture.
    return torch.nn.Sequential(  # RU: Слои выполняются по порядку. EN: Layers execute sequentially.
        torch.nn.Conv2d(1, 8, kernel_size=3, padding=1),  # RU: 8 карт 8×8, 80 параметров. EN: Eight 8×8 maps, 80 parameters.
        torch.nn.ReLU(),  # RU: Добавляем нелинейность. EN: Add nonlinearity.
        torch.nn.MaxPool2d(2),  # RU: Уменьшаем карты до 4×4. EN: Reduce maps to 4×4.
        torch.nn.Conv2d(8, 16, kernel_size=3, padding=1),  # RU: 16 карт, 1168 параметров. EN: Sixteen maps, 1168 parameters.
        torch.nn.ReLU(),  # RU: Нелинейность второго блока. EN: Nonlinearity of the second block.
        torch.nn.MaxPool2d(2),  # RU: Теперь 16 карт 2×2. EN: Now sixteen 2×2 maps.
        torch.nn.Flatten(),  # RU: Превращаем карты в 64 признака. EN: Flatten maps into 64 features.
        torch.nn.Linear(64, 10),  # RU: 10 logits, 650 параметров. EN: Ten logits, 650 parameters.
    )  # RU: Архитектура содержит 1898 параметров. EN: The architecture contains 1898 parameters.
model = build_cnn()  # RU: Создаём модель для проверки форм. EN: Create a model for shape inspection.
print(model)  # RU: Печатаем архитектуру. EN: Print the architecture.
assert sum(p.numel() for p in model.parameters()) == 1898  # RU: Проверяем число параметров. EN: Verify the parameter count.

**Мини-задание / Mini-task · 5–10 min.**
RU: Проведите батч из 7 train-изображений через каждый слой и запишите формы. Для каждого обучаемого слоя посчитайте веса и bias формулой и кодом. Затем сравните общее число параметров с MLP 64→32→10. Объясните, почему меньшая модель не обязана быть точнее. Сдать: таблицу слоёв/форм/параметров и вывод. Контрольные суммы CNN=1898, MLP=2410.
EN: Pass seven training images through each layer and record shapes. Count weights and biases analytically and in code for each trainable layer. Compare the total with an MLP 64→32→10. Explain why fewer parameters do not guarantee higher accuracy. Deliver a layer/shape/parameter table and a conclusion. Check totals: CNN=1898, MLP=2410.

In [ ]:
# Ваше решение / Your solution

## 6. Обучение и validation / Training and validation

RU: Две скорости обучения заранее: 0.001 и 0.003, Adam, 25 эпох, batch=64. Выбираем checkpoint и кандидата только по validation loss.

EN: Predeclared learning rates 0.001 and 0.003; Adam, 25 epochs, batch=64. Choose both checkpoint and candidate using validation loss only.

In [ ]:
def evaluate(model, inputs, targets):  # RU: Оценка без изменения весов. EN: Evaluation without changing weights.
    previous_mode = model.training  # RU: Запоминаем исходный режим. EN: Remember the original mode.
    model.eval()  # RU: Отключаем обучающее поведение слоёв. EN: Disable training-specific layer behavior.
    with torch.no_grad():  # RU: Не строим граф градиентов. EN: Do not build the gradient graph.
        logits = model(inputs)  # RU: Получаем logits. EN: Compute logits.
        loss = torch.nn.functional.cross_entropy(logits, targets).item()  # RU: Средняя cross-entropy. EN: Mean cross-entropy.
        accuracy = (logits.argmax(1) == targets).float().mean().item()  # RU: Доля правильных ответов. EN: Fraction of correct predictions.
    model.train(previous_mode)  # RU: Восстанавливаем исходный режим. EN: Restore the original mode.
    return {'loss': loss, 'accuracy': accuracy}  # RU: Возвращаем два скаляра. EN: Return two scalar metrics.

def fit(lr, epochs=25):  # RU: Обучение одного заранее заданного кандидата. EN: Train one predeclared candidate.
    torch.manual_seed(707)  # RU: Одинаковая начальная точка кандидатов. EN: Identical candidate initialization.
    model = build_cnn()  # RU: Создаём новую модель. EN: Create a fresh model.
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # RU: Выбираем Adam. EN: Use Adam optimization.
    rng = np.random.default_rng(707)  # RU: Повторяемое перемешивание реальных объектов. EN: Reproducible shuffling of real observations.
    best_loss = float('inf')  # RU: До первой эпохи лучшего loss нет. EN: No best loss before the first epoch.
    history = []  # RU: Накапливаем метрики эпох. EN: Collect epoch metrics.
    for epoch in range(epochs):  # RU: Выполняем заданное число эпох. EN: Run the fixed number of epochs.
        model.train()  # RU: Включаем обучение. EN: Enable training mode.
        order = rng.permutation(train_ids)  # RU: Перемешиваем только train. EN: Shuffle training data only.
        for start in range(0, len(order), 64):  # RU: Мини-батчи по 64. EN: Mini-batches of 64.
            batch = order[start:start + 64]  # RU: Индексы текущего батча. EN: Current batch indices.
            optimizer.zero_grad(set_to_none=True)  # RU: Сбрасываем старые градиенты. EN: Clear old gradients.
            loss = torch.nn.functional.cross_entropy(model(x[batch]), y[batch])  # RU: Loss по logits, без softmax заранее. EN: Loss from logits, without prior softmax.
            loss.backward()  # RU: Рассчитываем градиенты. EN: Compute gradients.
            optimizer.step()  # RU: Обновляем параметры. EN: Update parameters.
        train_result = evaluate(model, x[train_ids], y[train_ids])  # RU: Метрика train после эпохи. EN: Training metric after the epoch.
        val_result = evaluate(model, x[val_ids], y[val_ids])  # RU: Только validation выбирает checkpoint. EN: Only validation selects the checkpoint.
        history.append([train_result['loss'], val_result['loss']])  # RU: Сохраняем кривые. EN: Save learning curves.
        if val_result['loss'] < best_loss:  # RU: Обновляем только при строгом улучшении. EN: Update only on strict improvement.
            best_loss = val_result['loss']  # RU: Запоминаем лучший loss. EN: Remember the best loss.
            best_state = copy.deepcopy(model.state_dict())  # RU: Копируем, а не сохраняем ссылку. EN: Copy weights rather than retaining a reference.
            best_epoch = epoch + 1  # RU: Эпохи нумеруем с единицы. EN: Epoch numbering starts at one.
    model.load_state_dict(best_state)  # RU: Восстанавливаем лучший checkpoint. EN: Restore the best checkpoint.
    model.eval()  # RU: Подготавливаем модель к оценке. EN: Prepare the model for evaluation.
    return model, {'lr': lr, 'val_loss': best_loss, 'best_epoch': best_epoch, 'history': history}  # RU: Возвращаем модель и журнал. EN: Return the model and log.

def select_best(records):  # RU: Выбор по validation с первым индексом при равенстве. EN: Select by validation, taking the first tie.
    return min(range(len(records)), key=lambda i: records[i]['val_loss'])  # RU: Минимальный validation loss. EN: Minimum validation loss.
candidates = [fit(lr) for lr in [0.001, 0.003]]  # RU: Кандидаты заданы до просмотра test. EN: Candidates fixed before viewing test.
reports = [report for _, report in candidates]  # RU: Журналы validation. EN: Validation records.
winner = select_best(reports)  # RU: Выбираем минимальный validation loss. EN: Choose the minimum validation loss.
chosen, report = candidates[winner]  # RU: Фиксируем победителя. EN: Freeze the selected candidate.
for record in reports:  # RU: Показываем обе конфигурации. EN: Show both configurations.
    print({key: value for key, value in record.items() if key != 'history'})  # RU: Краткий журнал без массива эпох. EN: Compact log without epoch arrays.

**Мини-задание / Mini-task · 5–10 min.**
RU: Нарисуйте train и validation loss выбранного кандидата. Отметьте эпоху лучшего checkpoint. Докажите, что deepcopy(state_dict) не изменяется при изменении параметров отдельной копии модели. Проверьте, что evaluate восстанавливает режим train, не создаёт новых градиентов и не меняет веса. Сдать: график и assert-проверки. Test пока не открывать.
EN: Plot training and validation loss for the selected candidate and mark its best checkpoint. Prove a deep-copied state_dict remains unchanged when a separate model copy is modified. Verify evaluate restores training mode, creates no new gradients and preserves weights. Deliver the plot and assertions. Do not open test yet.

In [ ]:
# Ваше решение / Your solution

## 7. Test, feature maps и ошибки / Test, feature maps and errors

RU: После фиксации модели один раз считаем test. Карты промежуточных слоёв — отклики, а не доказательство причин решения.

EN: Evaluate test once after freezing the model. Intermediate maps are responses, not proof of why a decision was made.

In [ ]:
test_result = evaluate(chosen, x[test_ids], y[test_ids])  # RU: Единственная итоговая метрика test. EN: One final test metric.
with torch.no_grad():  # RU: Анализ без графа. EN: Analyze without a gradient graph.
    predicted = chosen(x[test_ids]).argmax(1).numpy()  # RU: Сохраняем предсказания для анализа. EN: Save predictions for analysis.
    feature_maps = chosen[:2](x[test_ids[:1]])[0].numpy()  # RU: Карты первого Conv–ReLU. EN: First Conv–ReLU feature maps.
print(test_result)  # RU: Печатаем результат. EN: Print the result.
ConfusionMatrixDisplay.from_predictions(labels[test_ids], predicted)  # RU: Матрица ошибок. EN: Confusion matrix.
plt.show()  # RU: Выводим матрицу. EN: Display the matrix.

**Мини-задание / Mini-task · 5–10 min.**
RU: Покажите исходную цифру и восемь feature maps первого слоя. Затем покажите до шести ошибочных test-изображений с истинной и предсказанной метками. Если ошибок меньше, не выдумывайте новые. Для трёх доступных ошибок предложите визуальную гипотезу, не меняя модель. Сдайте две фигуры, 3 гипотезы и 2 ограничения оценки. Не называйте карту объяснением причин без дополнительных исследований.
EN: Display the input digit and all eight first-layer feature maps. Show up to six misclassified test images with true and predicted labels. If fewer errors exist, do not invent examples. Give visual hypotheses for three available errors without changing the model. Deliver two figures, three hypotheses and two evaluation limitations. Do not claim a map establishes causality without further investigation.

In [ ]:
# Ваше решение / Your solution

## Источники и обновление / Sources and update

Сохранено из исходной темы CNN: convolution, stride, padding, pooling, dense; ручные операции NumPy. / Retained from the original CNN topic: convolution, stride, padding, pooling, dense and manual NumPy operations.

Обновление 2026: воспроизводимый CPU-эксперимент PyTorch на реальных UCI Digits, явное разбиение, checkpoint по validation, подробные мини-задания и построчные решения. Это не обучение ImageNet. / 2026 update: reproducible CPU PyTorch experiment on real UCI Digits, explicit splitting, validation checkpointing, detailed mini-tasks and line-commented solutions. This is not ImageNet training.

[Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html), [MaxPool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html), [training modes](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html).